# FIT5196 Assessment 2 - Solution Notebook Template

**Group:** Group030  
**Members:** Add names and student IDs.

Rename this notebook to `Group030_solution.ipynb` and replace the group placeholder.
This is an optional organisational scaffold, not a worked solution or a substitute
for the A2 specification and rubric. Add cells as needed and replace the prompts
with your work. Keep useful outputs visible in the submitted notebook.

The three data-cleansing parts (dirty, missing and outlier) belong to **Task 1**.
**Task 2** documents and checks those results. **Task 3** performs transformation
on a separate customer-level analysis copy built from your three cleaned CSVs.
Record audit evidence and validate repairs throughout Task 1; do not try to
reconstruct your decisions from memory at the end. You may reorganise the
workflow, but Task 3 must use your final cleaned outputs, not the raw batches.

An empty template running without errors does not demonstrate completion.


## 0. Configuration and environment

Extract your allocated ZIP into a directory such as `data/raw/Group030_A2/` beside the
notebook. Its three input CSVs, `warehouses.csv`, manifests, README and
`resources/` directory should be inside that directory. Change `GROUP_ID` and
the path settings to match your layout. Paths are relative to the working
directory from which the notebook is run.

`OUTPUT_DIR` below is the notebook's working directory, separate from the raw
input directory. Keep all configurable paths here and never overwrite inputs.


In [1]:
from pathlib import Path
import sys

GROUP_ID = "Group030"
WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = (
    WORKING_DIR
    if (WORKING_DIR / "data" / "raw").exists()
    else WORKING_DIR.parent
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw" / f"{GROUP_ID}_A2"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DIRTY_INPUT = DATA_DIR / f"{GROUP_ID}_dirty_data.csv"
MISSING_INPUT = DATA_DIR / f"{GROUP_ID}_missing_data.csv"
OUTLIER_INPUT = DATA_DIR / f"{GROUP_ID}_outlier_data.csv"
WAREHOUSES_INPUT = DATA_DIR / "warehouses.csv"
RESOURCE_CONTRACT = DATA_DIR / "resource_contract.json"
NLTK_DATA_DIR = DATA_DIR / "resources" / "nltk_data"

DIRTY_OUTPUT = OUTPUT_DIR / f"{GROUP_ID}_dirty_clean.csv"
MISSING_OUTPUT = OUTPUT_DIR / f"{GROUP_ID}_missing_clean.csv"
OUTLIER_OUTPUT = OUTPUT_DIR / f"{GROUP_ID}_outlier_clean.csv"
AUDIT_OUTPUT = OUTPUT_DIR / "audit_log.csv"

### 0.1 Dependencies and local resources

Record your Python version and dependencies, and supply `requirements.txt`.
Configure and verify the supplied frozen VADER resource as required by the spec.
Use a reproducible evaluation split where applicable. After dependencies are
installed, the submitted workflow must run offline without a Drive mount.


In [2]:
import platform
import sys

import matplotlib
import nltk
import numpy as np
import pandas as pd
import sklearn

from src.missing_cleaning import clean_missing_batch

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib:", matplotlib.__version__)
print("NLTK:", nltk.__version__)

Python: 3.12.3
pandas: 3.0.5
NumPy: 2.5.1
scikit-learn: 1.9.0
matplotlib: 3.11.1
NLTK: 3.9.2


The workflow uses only the package-local frozen VADER lexicon. The cleaning function checks both the lexicon SHA-256 value and the required NLTK version before constructing the analyser; it does not download resources. Seasonal train/test splits use `random_state=5196`. Exact dependency versions will be frozen in `requirements.txt` after the final clean-environment Run All.

## Task 1 - Data cleansing and explanation (15 marks)

### 1.0 Initial investigation and shared evidence

Read the three batches while keeping original values and source-batch identity.
Show diagnostic EDA that supports your choices: types, keys, missingness,
categories, distributions and relevant relationships. Explain which observations
are reliable enough to support calculations or model fitting.

Place reusable functions and shared evidence here or beside their first use.
Explain current-price inference and seasonal delivery-model training/evaluation
where they are used. Follow the published calculation, date, location and
VADER rules; do not infer answers from row positions or another group's data.


In [3]:
missing_original = pd.read_csv(
    MISSING_INPUT,
    dtype={"order_id": "string", "customer_id": "string"},
)

initial_missing_summary = (
    missing_original.isna().sum().loc[lambda values: values > 0]
    .rename("missing_values")
    .to_frame()
)
print("Missing batch shape:", missing_original.shape)
print("Unique order IDs:", missing_original["order_id"].nunique())
print("Affected rows:", int(missing_original.isna().any(axis=1).sum()))
print("Maximum missing fields in one row:", int(missing_original.isna().sum(axis=1).max()))
display(initial_missing_summary)

Missing batch shape: (500, 16)
Unique order IDs: 500
Affected rows: 80
Maximum missing fields in one row: 1


,missing_values
nearest_warehouse,10
order_price,10
delivery_charges,10
order_total,10
season,10
is_expedited_delivery,10
distance_to_nearest_warehouse,10
is_happy_customer,10


The missing batch contains 500 unique orders and 80 omissions across 80 different rows, with no row missing more than one field. Eight fields each have ten omissions. Under the supplied batch guarantee, every observed value in this batch is reliable. This makes the batch suitable for deterministic reconstruction and for fitting the first seasonal delivery models on rows where the target and all three predictors are observed.

### 1.1 Dirty data - diagnose and repair

Identify the affected field using evidence and the batch guarantees. Explain
how you resolve checks that disagree because of one corrupted field. Correct
only affected information, preserve valid values and retain all 500 order IDs.
Record actual changes and the evidence needed for the audit log as you work.


In [4]:
# Add dirty-batch diagnosis, justified repairs and checks.


*Replace this prompt with your evidence, interpretation and decisions.*


### 1.2 Missing data - recover omitted values

Report missingness by field and justify each treatment. Use calculable
relationships where available. For model-based estimates, explain your training
evidence and evaluation. Preserve observed values and all 500 order IDs.
Record actual imputations and their supporting evidence.


In [5]:
missing_result = clean_missing_batch(
    missing_input=MISSING_INPUT,
    warehouses_input=WAREHOUSES_INPUT,
    resource_contract=RESOURCE_CONTRACT,
    nltk_data_dir=NLTK_DATA_DIR,
    missing_output=MISSING_OUTPUT,
    audit_output=AUDIT_OUTPUT,
)

print("Product-price evidence")
display(missing_result.price_diagnostics)
display(missing_result.product_prices)

print("Seasonal delivery-model held-out evidence")
display(missing_result.model_diagnostics)

print("Imputation summary")
display(missing_result.imputation_summary)

print("Missing-output read-back validation")
display(missing_result.validation)

Product-price evidence


,observed_equations,training_equations,heldout_equations,training_rank,product_count,heldout_max_absolute_error,all_equations_max_absolute_error
0,490,392,98,12,12,0.0,0.0


,product,unit_price
0,Candle Arc 420,2742.91
1,Candle Bloom 316,698.58
2,Candle Bloom 838,1098.41
3,Candle Edge 572,1340.90
4,Candle Lumen 778,1371.48
5,Candle Shift 232,801.49
6,Vela Atlas 863,3657.74
7,Vela Echo 547,899.34
8,Vela Echo 583,1072.62
9,Vela Halo 137,769.91


Seasonal delivery-model held-out evidence


,season,reliable_observations,training_observations,heldout_observations,heldout_r_squared,heldout_mae,intercept,distance_coefficient,expedited_coefficient,happy_coefficient
0,Summer,125,100,25,1.000000,0.002419,9.999191,0.700247,9.999835,-2.000375
1,Autumn,103,82,21,1.000000,0.002164,9.996941,0.780000,11.000129,-2.996824
2,Winter,116,92,24,1.000000,0.002822,14.001254,0.849878,13.000287,-4.000432
3,Spring,106,84,22,0.999999,0.002197,9.999172,0.730389,10.499802,-2.500837


Imputation summary


,field_name,confidence,imputed_values
0,delivery_charges,high,10
1,distance_to_nearest_warehouse,high,10
2,is_expedited_delivery,high,10
3,is_happy_customer,high,10
4,nearest_warehouse,high,10
5,order_price,high,10
6,order_total,high,10
7,season,high,10


Missing-output read-back validation


,check,observed_result,passed,interpretation
0,schema_and_column_order,"[order_id, customer_id, date, nearest_warehous...",True,Output must use the exact original 16-column s...
1,row_count,500,True,All missing-batch orders retained.
2,unique_order_ids,500,True,One row remains for each order.
3,preserved_order_id_set,"{'missing': 0, 'unexpected': 0}",True,No order IDs were lost or introduced.
4,required_completeness,0,True,No empty fields remain.
5,shopping_cart_structure,True,True,All carts are valid JSON arrays.
6,order_price_relationship,0.0,True,Cart equations reproduce order prices.
7,date_season_relationship,0,True,Dates map to Australian seasons.
8,nearest_warehouse_relationship,0,True,Nearest warehouse agrees with coordinates.
9,distance_relationship,0.0,True,Haversine distances agree to four decimals.


#### Missing-value treatment and evidence

All 80 omissions were reconstructed without changing any observed source cell.

- `season`: published Australian month-to-season mapping.
- `nearest_warehouse`: minimum Haversine distance across the three supplied warehouses.
- `distance_to_nearest_warehouse`: Haversine distance with a 6,371 km Earth radius, rounded to four decimals.
- `order_price`: the published per-item rounding sequence using current unit prices inferred from reliable cart equations. A reproducible 80/20 equation split retained full rank 12; all 98 held-out equations and all 490 observed equations had maximum absolute error AUD 0.00.
- `order_total`: the published discounted-item-price plus delivery formula, rounded only at the end.
- `is_happy_customer`: verified frozen VADER lexicon, the `compound >= 0.05` rule, and the exact no-review sentinel rule.
- `delivery_charges`: separate seasonal linear regressions using distance, expedited status and happiness. The models used 450 rows with observed response and predictors. Held-out R-squared was at least 0.999999 and MAE was below AUD 0.003 in every season.
- `is_expedited_delivery`: evaluated both 0 and 1 in the applicable seasonal model and selected the state with the smaller absolute delivery-charge residual.

All replacements are logged with high confidence because deterministic fields follow exact published rules, while model-based fields have near-perfect held-out evidence and clear residual separation. Ten model-imputed delivery charges produce at most a one-cent difference when recomputing a small number of already-valid observed totals. In accordance with the specification, those valid totals were preserved rather than overwritten to force agreement with an estimate.

### 1.3 Outlier data - identify and remove anomalous rows

Use seasonal delivery-charge models, residual diagnostics and a justified
decision rule. Show held-out sample sizes, R-squared and an absolute-error
measure for each season. Explain how you distinguish anomalous charges from
unusual but valid orders. Remove identified outlier rows rather than replacing
their charges; preserve every retained row's valid values. Record removed IDs
and the evidence for removal.


In [6]:
# Add outlier diagnosis, justified row removal and retained-row checks.


*Replace this prompt with your evidence, interpretation and decisions.*


### 1.4 Export the three cleaned CSVs

Write the three specified output files using the original 16-column schema,
column order and units, without a pandas index. Keep dirty and missing outputs
at 500 rows each; the outlier output contains only retained original IDs.
The exported CSVs must not contain Task 3 scaled or transformed values.


In [7]:
# Write DIRTY_OUTPUT, MISSING_OUTPUT and OUTLIER_OUTPUT from your workflow.


## Task 2 - Audit trail and validation (2 marks)

### 2.1 Generate the field-level audit log

Export `audit_log.csv` from the changes recorded by your workflow, using this
column order:

`source_file`, `record_key`, `field_name`, `issue_type`, `original_value`,
`corrected_value`, `method`, `evidence`, `confidence`.

Use one row per changed field and one row per removed outlier, following the
spec's value conventions. Define your confidence levels and give traceable
evidence references. Do not log unchanged cells, harmless formatting differences
or Task 3 analytical transformations as cleaning corrections.


In [8]:
# Generate AUDIT_OUTPUT from recorded changes and removals.


*Replace this prompt with your evidence, interpretation and decisions.*


### 2.2 Read back and validate the submitted outputs

Re-read all three cleaned CSVs and the audit log from disk. Show executable
checks covering the schema, column order, types and completeness; ID and row
requirements; cart/current-price, calendar, warehouse/location, happiness and
arithmetic relationships; seasonal fit and retained residuals; preservation of
valid information; and agreement between actual changes and the audit log.

For each check, record the observed result, pass/fail outcome and interpretation
or resolution. Explain unresolved issues honestly. Reuse earlier evidence by
clear cross-reference rather than repeating it. No single passing check proves
that every output is correct.


In [9]:
# Current read-back validation evidence for the completed missing batch.
# This section will be expanded after the dirty and outlier outputs are generated.
display(missing_result.validation)

,check,observed_result,passed,interpretation
0,schema_and_column_order,"[order_id, customer_id, date, nearest_warehous...",True,Output must use the exact original 16-column s...
1,row_count,500,True,All missing-batch orders retained.
2,unique_order_ids,500,True,One row remains for each order.
3,preserved_order_id_set,"{'missing': 0, 'unexpected': 0}",True,No order IDs were lost or introduced.
4,required_completeness,0,True,No empty fields remain.
5,shopping_cart_structure,True,True,All carts are valid JSON arrays.
6,order_price_relationship,0.0,True,Cart equations reproduce order prices.
7,date_season_relationship,0,True,Dates map to Australian seasons.
8,nearest_warehouse_relationship,0,True,Nearest warehouse agrees with coordinates.
9,distance_relationship,0.0,True,Haversine distances agree to four decimals.


At this stage, every missing-batch validation check passes: exact schema and column order, 500 unique preserved IDs, complete values, valid carts, cart-price equations, date-season mapping, nearest-warehouse and Haversine relationships, VADER happiness, monetary tolerances, preservation of all originally observed cells, and one-to-one reconciliation of the 80 source omissions with the 80 audit rows. Final Task 2 evidence will additionally reconcile dirty repairs and removed outlier IDs.

## Task 3 - Customer profiling and transformation (3 marks)

Begin this analysis using the final three cleaned CSVs from Tasks 1-2. Work on
a separate analysis copy; do not change the cleaning outputs or audit log.
The purpose is possible future distance-based exploratory customer segmentation.
A clustering/prediction model or dashboard is not required.

### 3.1 Build the customer-level analysis copy

Read and combine your three cleaned outputs, retaining source-batch identity.
Report retained counts by batch, distinct order count and customer count.
Do not restore removed outliers, append A1 data or use another group's data.
Explain any unresolved input issues and the limits of this retained A2 sample.

Aggregate to one row per `customer_id` and select **three** of the following:

| Attribute | Definition over retained orders |
|---|---|
| `order_count` | Count of distinct `order_id` values |
| `total_paid` | Sum of `order_total` |
| `mean_order_value` | Mean of `order_total` |
| `mean_delivery_distance` | Mean of `distance_to_nearest_warehouse` |

Do not transform IDs. If selecting count, total and mean together, discuss their
algebraic dependence and possible repeated weighting of spending information.
These are sample profiles, not lifetime customer histories.


In [10]:
# Build customer profiles from the three submitted cleaned CSVs.


*Replace this prompt with your evidence, interpretation and decisions.*


### 3.2 Original scales, distributions and relationships

Keep an untransformed baseline. Use relevant numerical summaries and/or plots
to diagnose units, scales, distributions and relationships among your selected
attributes. Interpret their implications for distance-based customer comparison.


In [11]:
# Add original-profile diagnostics and preserve the baseline.


*Replace this prompt with your evidence, interpretation and decisions.*


### 3.3 Compare standardisation and min-max scaling

Compare both z-score standardisation and min-max scaling on all three selected
attributes. Show their actual effects and explain relevant domain constraints,
including constant values if encountered. Low skewness alone does not establish
that a feature needs no scaling.


In [12]:
# Implement and compare the two required scaling approaches.


*Replace this prompt with your evidence, interpretation and decisions.*


### 3.4 Compare an applicable nonlinear transformation

Compare at least one applicable nonlinear transformation with original values
for at least one selected attribute. Explain its domain requirements, observed
effect and suitability. State the order if combining it with scaling.
Normality, lower skewness or higher R-squared is not a universal success rule.


In [13]:
# Implement an applicable nonlinear comparison and examine its effect.


*Replace this prompt with your evidence, interpretation and decisions.*


### 3.5 Recommendations, boundaries and limitations

Complete the table with one row per selected attribute. State nonlinear
treatment (or none) and scaling (or none) separately, and their order if combined.
Justify decisions using your comparisons rather than a fixed preferred answer.
If retaining an attribute without scaling, explain how differences in units and
scale are handled or justified for this purpose.

| Attribute | Methods compared | Observed effect / evidence reference | Recommended treatment (nonlinear; scaling; order) | Justification / limitation |
|---|---|---|---|---|
| [Selected attribute 1] | | | | |
| [Selected attribute 2] | | | | |
| [Selected attribute 3] | | | | |

Discuss relevant effects on relative distances, extreme observations,
interpretability and/or redundant features. Do not claim the comparison proves
that meaningful customer clusters exist. If you optionally evaluate a held-out
model, fit transformation parameters on its training data only. Keep the profiles
and transformed features in this notebook; no extra CSV or report is required.


*Replace this prompt with your evidence, interpretation and decisions.*


## Final reproducibility and submission check

This is a submission check, not an additional assessed task.

- Restart the kernel and Run All; retain the completed notebook's useful outputs.
- Check that this run reproduces the three cleaned CSVs, `audit_log.csv` and
  the Task 3 comparisons. Keep originals unchanged.
- Include matching `Group030_solution.py`, imported helpers and `requirements.txt`.
- Put the required files in `Group030_A2_submission/` and ZIP that folder as
  `Group030_A2_submission.zip`, following the spec's checklist.
- Do not submit the raw inputs, frozen lexicon, environment or temporary files.
- Include the signed `Group030_AI_declaration.pdf`, including when no AI was
  used, and applicable complete chat exports with `AI_records/Group030_AI_index.pdf`.
- Use English for all A2-related AI conversations, chat exports, declarations
  and indexes.
- Cite external code and sources. This notebook does not replace the separate
  declaration or applicable AI records.

No separate EDA PDF, fixed figure count or ten-page report limit applies to A2.

### References and final notes

*Add references and any concise final reproducibility notes here.*
